In [ ]:
### Definição das perguntas de negócio:
# 1) Clientes com maior renda anual apresentam menores taxas de inadiplência?
# 2) O tipo de moradia ou o estado civil têm relação direta com o risco de atraso de pagamento?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 130)

df_clientes = pd.read_csv('../dados/application_record.csv')
df_credito = pd.read_csv('../dados/credit_record.csv')

display(df_clientes.shape)
display(df_credito.shape)

In [ ]:
# Correção da padronização das colunas originais

df_clientes['ID'] = pd.to_numeric(
    df_clientes['ID'], errors='coerce'
).astype('Int64')

df_credito['ID'] = pd.to_numeric(
    df_credito['ID'], errors='coerce'
).astype('Int64')

df_credito['STATUS'] = (
    df_credito['STATUS']
    .astype('string')
    .str.strip()
)

# Remover registros sem identificador
df_clientes = df_clientes.dropna(subset=['ID'])
df_credito = df_credito.dropna(subset=['ID'])

print('Colunas dos clientes:', df_clientes.columns.tolist())
print('Colunas do crédito:', df_credito.columns.tolist())
print('Tipos das chaves:', df_clientes['ID'].dtype, df_credito['ID'].dtype)

In [ ]:
display(df_clientes.head())

In [ ]:
display(df_credito.head())

In [ ]:
novos_nomes = {
    'ID': 'id_cliente',
    'CODE_GENDER': 'genero',
    'FLAG_OWN_CAR': 'possui_carro',
    'FLAG_OWN_REALTY': 'possui_imovel',
    'CNT_CHILDREN': 'qtd_filhos',
    'AMT_INCOME_TOTAL': 'renda_anual',
    'NAME_INCOME_TYPE': 'tipo_renda',
    'NAME_EDUCATION_TYPE': 'escolaridade',
    'NAME_FAMILY_STATUS': 'estado_civil',
    'NAME_HOUSING_TYPE': 'tipo_moradia',
    'DAYS_BIRTH': 'dias_nascimento',
    'DAYS_EMPLOYED': 'dias_empregado',
    'FLAG_MOBIL': 'possui_celular',
    'FLAG_WORK_PHONE': 'possui_tel_trabalho',
    'FLAG_PHONE': 'possui_tel_fixo',
    'FLAG_EMAIL': 'possui_email',
    'OCCUPATION_TYPE': 'ocupacao',
    'CNT_FAM_MEMBERS': 'qtd_membros_familia'
}

In [ ]:
df_clientes = df_clientes.rename(columns=novos_nomes)
df_clientes.head()

In [ ]:
df_credito = df_credito.rename(columns={
    'ID': 'ID_CLIENTE',
    'MONTHS_BALANCE': 'MES_REFERENCIA',
    'STATUS': 'STATUS_PAGAMENTO'
})
display(df_credito.head())

In [ ]:
df_clientes.info()

In [ ]:
display(df_clientes['ocupacao'].isna().sum())
print(df_clientes['ocupacao'].unique())

In [ ]:
print(df_clientes[df_clientes['ocupacao'].isna()]['tipo_renda'].value_counts())

In [ ]:
condicoes = [
    (df_clientes["ocupacao"].isna())
    & (df_clientes["tipo_renda"] == "Pensioner"),
    (df_clientes["ocupacao"].isna()),
]

escolhas = ["Pensioner", "Not Specified"]

df_clientes["ocupacao"] = np.select(
    condicoes, escolhas, default=df_clientes["ocupacao"]
)

In [ ]:
display(df_clientes['ocupacao'].isna().sum())

In [ ]:
df_clientes.duplicated().sum()

In [ ]:
df_credito.info()

In [ ]:
display(df_credito.head())
display(df_credito.tail())

In [ ]:
df_credito.duplicated().sum()

In [ ]:
df_credito['STATUS_PAGAMENTO'].unique()

In [ ]:
# ==============================================================================
# DICIONÁRIO DE STATUS DE PAGAMENTO / ATRASO (STATUS)
# ------------------------------------------------------------------------------
# C | Pago em dia / Empréstimo quitado no mês
# X | Nenhum empréstimo ativo no mês
# 0 | 1 a 29 dias de atraso
# 1 | 30 a 59 dias de atraso
# 2 | 60 a 89 dias de atraso
# 3 | 90 a 119 dias de atraso
# 4 | 120 a 149 dias de atraso
# 5 | 150+ dias de atraso ou dívida baixada/prejuízo
# ==============================================================================

In [ ]:
# Tratamento da tabela cadastral

df_clientes['idade'] = (-df_clientes['dias_nascimento'] / 365.25).astype(int)
df_clientes['anos_emprego'] = (
    -df_clientes['dias_empregado'] / 365.25
).clip(lower=0).round(1)
df_clientes['ocupacao'] = df_clientes['ocupacao'].fillna('Não informada')

In [ ]:
# Selecionando features relevantes

cols_selecionadas = [
    'id_cliente',
    'genero',
    'possui_carro',
    'possui_imovel',
    'renda_anual',
    'tipo_renda',
    'escolaridade',
    'estado_civil',
    'tipo_moradia',
    'idade',
    'anos_emprego',
    'qtd_membros_familia',
]
df_clientes_limpa = df_clientes[cols_selecionadas].copy()

In [ ]:
# Definindo mau pagador se atrasou 30+ dias (status 1 a 5) em qualquer mês
inadimplente = ['1', '2', '3', '4', '5']
df_credito['inadimplente'] = df_credito['STATUS_PAGAMENTO'].isin(inadimplente).astype(int)

In [ ]:
df_credito['inadimplente']

In [ ]:
# Agrupando por cliente (1 = já teve atraso grave, 0 = nunca atrasou >30 dias)
df_target = df_credito.groupby("ID_CLIENTE")["inadimplente"].max().reset_index().rename(columns={'ID_CLIENTE':'id_cliente'})

In [ ]:
df_target

In [ ]:
# Cruzamento dos dados
df_final = pd.merge(df_clientes_limpa, df_target, on='id_cliente', how='inner')

In [ ]:
display(df_final.head())

In [ ]:
print('--- PERGUNTA 1: Faixa de Renda vs Inadimplência ---')
df_final['faixa_renda'] = pd.qcut(
    df_final['renda_anual'], q=2, labels=['Menor Renda', 'Maior Renda']
)
print(
    df_final.groupby('faixa_renda', observed=False)['inadimplente']
    .agg(total_clientes='count', taxa_inadimplencia='mean')
    .reset_index()
)

In [ ]:
# Renda isolada raramente é o fator mais determinante para atrasos. Clientes de maior renda tomam limites mais altos e alavancam mais faturas; variáveis como estabilidade no emprego (anos_emprego) e faixa etária (idade) tendem a ser preditores muito mais fortes de adimplência do que o valor nominal da renda.

In [ ]:
print('--- PERGUNTA 1: Faixa de Renda vs Percentual de Inadimplência ---')

if 'df_final' not in globals():
    raise RuntimeError(
        'df_final ainda não foi criado. Execute as células anteriores '
        'até o cruzamento dos dados antes desta análise.'
    )

colunas_necessarias = {'renda_anual', 'inadimplente'}
colunas_ausentes = colunas_necessarias.difference(df_final.columns)
if colunas_ausentes:
    raise ValueError(
        f'Colunas ausentes em df_final: {sorted(colunas_ausentes)}. '
        'Verifique a preparação dos dados.'
    )

# Cria cinco faixas com quantidades de clientes aproximadamente equivalentes.
df_faixas_renda = df_final[['renda_anual', 'inadimplente']].dropna().copy()
df_faixas_renda['faixa_renda'] = pd.qcut(
    df_faixas_renda['renda_anual'],
    q=5,
    labels=['Muito baixa', 'Baixa', 'Média', 'Alta', 'Muito alta'],
    duplicates='drop'
)

resultado_faixas = (
    df_faixas_renda.groupby('faixa_renda', observed=False)['inadimplente']
    .agg(
        total_clientes='count',
        total_inadimplentes='sum',
        percentual_inadimplencia='mean'
    )
    .reset_index()
)
resultado_faixas['percentual_inadimplencia'] = (
    resultado_faixas['percentual_inadimplencia'] * 100
).round(2)

print('Percentual de inadimplência por faixa de renda:')
display(resultado_faixas)

plt.figure(figsize=(6, 3.5))
ax = sns.barplot(
    data=resultado_faixas,
    x='faixa_renda',
    y='percentual_inadimplencia',
    color='steelblue'
)

for barra in ax.patches:
    altura = barra.get_height()
    ax.annotate(
        f'{altura:.2f}%',
        (barra.get_x() + barra.get_width() / 2, altura),
        ha='center',
        va='bottom',
        xytext=(0, 5),
        textcoords='offset points'
    )

plt.title('Percentual de Inadimplência por Faixa de Renda')
plt.xlabel('Faixa de renda')
plt.ylabel('Percentual de inadimplência (%)')
plt.ylim(0, resultado_faixas['percentual_inadimplencia'].max() * 1.2)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()